# Temporal NTL Animation — Electrification Change 2015–2023

**Author:** Bouchra Daddaoui  
**Repository:** viirs-electrification-ml  

Animated choropleth showing how VIIRS nighttime light radiance evolved spatially
across tiles from 2015 to 2023 — illustrating the geographic spread of electrification
and the persistence of energy poverty in specific regions.

**Methods:**
- Spatially correlated NTL synthetic panel (tile-level, 2015–2023)
- Annual growth parameterised from VIIRS empirical trends
- Animated GIF via `matplotlib.animation.FuncAnimation`
- Plotly animated choropleth for interactive exploration

**Outputs:**
- `figures/ntl_animation_brazil.gif`
- `figures/ntl_animation_china.gif`
- `figures/ntl_animation_morocco.gif`
- `figures/ntl_animation_combined.gif`
- `figures/ntl_temporal_plotly.html` — interactive slider

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import plotly.express as px
from shapely.geometry import box, mapping
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)

YEARS = list(range(2015, 2024))
print(f'Years: {YEARS[0]}–{YEARS[-1]}')

## 1. Spatially Correlated Temporal Panel

Each tile has a NTL trajectory over 2015–2023. NTL evolves as:
- Spatially correlated base NTL in 2015
- Tile-specific annual growth rates (log-normal, spatially smoothed)
- Realistic shock years (China: COVID dip 2020; Morocco: rural electrification push 2017)

In [ ]:
def generate_temporal_tiles(country, n_tiles, bbox, ntl_mean_2015, ntl_std,
                             mean_growth, growth_std, shocks, seed):
    """
    Generate a GeoDataFrame with NTL values for each tile and each year.
    shocks: dict {year: additive_shock} applied to all tiles.
    """
    rng = np.random.default_rng(seed)
    minx, miny, maxx, maxy = bbox
    side = int(np.sqrt(n_tiles))
    xs = np.linspace(minx, maxx, side + 1)
    ys = np.linspace(miny, maxy, side + 1)

    geoms, tile_ids = [], []
    for i in range(side):
        for j in range(side):
            geoms.append(box(xs[j], ys[i], xs[j+1], ys[i+1]))
            tile_ids.append(f'{country[:3].upper()}_{i:02d}_{j:02d}')

    n = len(geoms)
    centroids = np.array([[g.centroid.x, g.centroid.y] for g in geoms])

    # Spatially correlated base NTL in 2015
    dists = np.linalg.norm(centroids[:, None] - centroids[None, :], axis=-1)
    cov = ntl_std**2 * np.exp(-dists / (0.3 * (maxx - minx)))
    ntl_base = rng.multivariate_normal(np.full(n, ntl_mean_2015), cov)
    ntl_base = np.clip(ntl_base, 0.5, None)

    # Tile-specific growth rates (spatially smoothed via same covariance)
    growth_cov = growth_std**2 * np.exp(-dists / (0.5 * (maxx - minx)))
    tile_growth = rng.multivariate_normal(np.full(n, mean_growth), growth_cov)

    # Build yearly NTL matrix
    yearly_ntl = {}
    ntl_curr = ntl_base.copy()
    for yr in YEARS:
        shock = shocks.get(yr, 0)
        ntl_curr = ntl_curr + tile_growth + rng.normal(0, 0.3, n) + shock
        ntl_curr = np.clip(ntl_curr, 0, None)
        yearly_ntl[yr] = ntl_curr.copy()

    gdf_base = gpd.GeoDataFrame({
        'tile_id': tile_ids,
        'country': country,
        'lon': centroids[:, 0],
        'lat': centroids[:, 1],
    }, geometry=geoms, crs='EPSG:4326')

    return gdf_base, yearly_ntl


country_params = [
    dict(country='Brazil',  n_tiles=100, bbox=(-48,-23,-43,-18),
         ntl_mean_2015=9.5,  ntl_std=6.0,  mean_growth=0.42, growth_std=0.15, shocks={}, seed=1),
    dict(country='China',   n_tiles=100, bbox=(116,29,122,33),
         ntl_mean_2015=21.0, ntl_std=12.0, mean_growth=1.85, growth_std=0.40,
         shocks={2020: -3.1}, seed=2),
    dict(country='Morocco', n_tiles=100, bbox=(-17.1,20.8,-1.0,35.9),
         ntl_mean_2015=6.2,  ntl_std=4.0,  mean_growth=0.38, growth_std=0.12,
         shocks={2017: 1.5}, seed=3),
]

panel = {}
for params in country_params:
    c = params['country']
    gdf_base, yearly_ntl = generate_temporal_tiles(**params)
    panel[c] = (gdf_base, yearly_ntl)
    ntl_2023 = yearly_ntl[2023]
    print(f'{c}: 2015 mean={yearly_ntl[2015].mean():.1f} → 2023 mean={ntl_2023.mean():.1f}')

## 2. Animated GIF — Per Country

NTL choropleth animated frame-by-frame, 2015→2023.  
Colour scale fixed to the full time-range extent for comparability across years.

In [ ]:
def make_country_gif(country, gdf_base, yearly_ntl, save_path, fps=2):
    # Fix colour limits to the full range across all years
    all_vals = np.concatenate(list(yearly_ntl.values()))
    vmin, vmax = np.percentile(all_vals, 2), np.percentile(all_vals, 98)
    cmap = plt.cm.plasma
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    fig, ax = plt.subplots(figsize=(6, 5))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('NTL Radiance (nW/cm²/sr)', color='white', fontsize=8)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    title = ax.set_title('', color='white', fontsize=12, fontweight='bold', pad=10)
    ax.set_axis_off()

    patches = [None]

    def update(yr):
        if patches[0] is not None:
            patches[0].remove()
        ntl = yearly_ntl[yr]
        colours = cmap(norm(ntl))
        gdf_yr = gdf_base.copy()
        patches[0] = gdf_yr.plot(color=colours, edgecolor='#333333',
                                  linewidth=0.2, ax=ax).collections[-1] \
                     if hasattr(gdf_yr.plot(color=colours, edgecolor='#333333',
                                            linewidth=0.2, ax=ax), 'collections') \
                     else ax.collections[-1]
        title.set_text(f'{country} — VIIRS NTL {yr}')
        return patches[0], title

    # Simpler: rebuild each frame
    frames_data = []
    for yr in YEARS:
        ntl = yearly_ntl[yr]
        frames_data.append((yr, ntl))

    def animate(frame_idx):
        ax.cla()
        ax.set_facecolor('#0d1117')
        ax.set_axis_off()
        yr, ntl = frames_data[frame_idx]
        colours = cmap(norm(ntl))
        gdf_base.plot(color=colours, edgecolor='#1a1a2e', linewidth=0.3, ax=ax)
        ax.set_title(f'{country} — VIIRS NTL {yr}',
                     color='white', fontsize=11, fontweight='bold')

    ani = animation.FuncAnimation(
        fig, animate, frames=len(YEARS), interval=500, repeat=True,
    )
    ani.save(str(save_path), writer='pillow', fps=fps, dpi=100)
    plt.close(fig)
    print(f'Saved: {save_path}')


for country, (gdf_base, yearly_ntl) in panel.items():
    make_country_gif(
        country, gdf_base, yearly_ntl,
        save_path=FIGURES / f'ntl_animation_{country.lower()}.gif',
        fps=1,
    )

## 3. Combined 3-Country Animation

Side-by-side panel showing all three countries evolving simultaneously.

In [ ]:
countries = list(panel.keys())
cmaps_per = {'Brazil': plt.cm.YlOrRd, 'China': plt.cm.PuBu, 'Morocco': plt.cm.YlGn}

# Compute per-country fixed colour limits
vlims = {}
for c, (_, yearly_ntl) in panel.items():
    all_vals = np.concatenate(list(yearly_ntl.values()))
    vlims[c] = (np.percentile(all_vals, 2), np.percentile(all_vals, 98))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.set_axis_off()

suptitle = fig.suptitle('', color='white', fontsize=13, fontweight='bold', y=1.02)

def animate_combined(frame_idx):
    yr = YEARS[frame_idx]
    for ax, country in zip(axes, countries):
        ax.cla()
        ax.set_facecolor('#0d1117')
        ax.set_axis_off()
        gdf_base, yearly_ntl = panel[country]
        ntl = yearly_ntl[yr]
        cmap = cmaps_per[country]
        norm = mcolors.Normalize(*vlims[country])
        colours = cmap(norm(ntl))
        gdf_base.plot(color=colours, edgecolor='#1a1a2e', linewidth=0.3, ax=ax)
        ax.set_title(country, color='white', fontsize=11, fontweight='bold')
    suptitle.set_text(f'VIIRS Nighttime Light Radiance — {yr}')

ani_combined = animation.FuncAnimation(
    fig, animate_combined, frames=len(YEARS), interval=700, repeat=True,
)
out = FIGURES / 'ntl_animation_combined.gif'
ani_combined.save(str(out), writer='pillow', fps=1, dpi=100)
plt.close(fig)
print(f'Saved: {out}')

## 4. Plotly Animated Choropleth (Interactive Slider)

Long-format dataframe with a year slider — allows scrubbing through the timeline interactively.

In [ ]:
import json
import plotly.graph_objects as go

frames_list = []
for country, (gdf_base, yearly_ntl) in panel.items():
    for yr in YEARS:
        ntl = yearly_ntl[yr]
        gdf_yr = gdf_base.copy()
        gdf_yr['ntl'] = ntl
        gdf_yr['year'] = yr
        gdf_yr['lon'] = gdf_yr.geometry.centroid.x
        gdf_yr['lat'] = gdf_yr.geometry.centroid.y
        frames_list.append(gdf_yr[['tile_id', 'country', 'year', 'lon', 'lat', 'ntl']])

long_df = pd.concat(frames_list, ignore_index=True)

fig_plotly = px.scatter_geo(
    long_df,
    lat='lat', lon='lon',
    color='ntl',
    size='ntl',
    size_max=15,
    animation_frame='year',
    color_continuous_scale='Plasma',
    hover_name='tile_id',
    hover_data={'country': True, 'ntl': ':.2f', 'lat': False, 'lon': False},
    title='VIIRS NTL Radiance 2015–2023 (Interactive)',
    labels={'ntl': 'NTL (nW/cm²/sr)'},
    projection='natural earth',
)

fig_plotly.update_layout(
    paper_bgcolor='#0d1117',
    font_color='white',
    geo=dict(bgcolor='#0d1117', landcolor='#1c2738',
             showland=True, showocean=True, oceancolor='#0a1628'),
)

out_html = FIGURES / 'ntl_temporal_plotly.html'
fig_plotly.write_html(str(out_html))
print(f'Saved: {out_html}')
print()
print('Open figures/ntl_temporal_plotly.html — use the year slider to animate.')

## 5. Tile-Level Growth Rate Map

Maps the Theil-Sen slope of NTL for each tile — identifies which areas are electrifying
fastest and which are stagnating or declining.

In [ ]:
from scipy import stats as scipy_stats

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')

for ax, (country, (gdf_base, yearly_ntl)) in zip(axes, panel.items()):
    n_tiles = len(gdf_base)
    slopes = np.zeros(n_tiles)
    ntl_matrix = np.array([yearly_ntl[yr] for yr in YEARS])  # shape (n_years, n_tiles)

    for t in range(n_tiles):
        slope, _, _, _, _ = scipy_stats.linregress(YEARS, ntl_matrix[:, t])
        slopes[t] = slope

    gdf_plot = gdf_base.copy()
    gdf_plot['growth_rate'] = slopes

    vmax = np.abs(slopes).max()
    gdf_plot.plot(
        column='growth_rate', cmap='RdYlGn', vmin=-vmax, vmax=vmax,
        edgecolor='#333', linewidth=0.2, legend=True,
        legend_kwds={'label': 'NTL growth (units/yr)', 'shrink': 0.6},
        ax=ax,
    )
    ax.set_facecolor('#0d1117')
    ax.set_title(f'{country}\nTile-Level NTL Growth Rate 2015–2023',
                 color='white', fontsize=10, fontweight='bold')
    ax.set_axis_off()

fig.suptitle('Tile-Level Electrification Growth Rates (OLS slope, nW/cm²/sr per year)',
             color='white', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES / 'ntl_growth_rate_map.png', dpi=180, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved → figures/ntl_growth_rate_map.png')